# Processor API Test Notebook

이 노트북은 `src/app/api/routers/processors.py`의 모든 엔드포인트를 테스트합니다.

## 테스트 대상 엔드포인트
1. `POST /processors/summarize` - 아티클 요약
2. `POST /processors/evaluate` - 중요도 평가
3. `POST /processors/classify` - 카테고리 분류
4. `POST /processors/process` - 전체 파이프라인 처리
5. `POST /processors/batch-process` - 배치 처리
6. `POST /processors/statistics` - 통계 계산

## 사전 요구사항
- FastAPI 서버가 실행 중이어야 합니다: `uvicorn src.app.api.main:app --reload`
- `.env` 파일에 `OPENAI_API_KEY`가 설정되어 있어야 합니다
- Qdrant가 실행 중이어야 합니다: `docker-compose up -d`

In [ ]:
import requests
import json
from pprint import pprint
import time

In [ ]:
# API 서버 설정
BASE_URL = "http://localhost:8000"
PROCESSORS_URL = f"{BASE_URL}/processors"

# 샘플 데이터
sample_paper = {
    "title": "Attention Is All You Need",
    "content": """
    We propose a new simple network architecture, the Transformer,
    based solely on attention mechanisms, dispensing with recurrence
    and convolutions entirely. Experiments on two machine translation
    tasks show these models to be superior in quality while being
    more parallelizable and requiring significantly less time to train.
    Our model achieves 28.4 BLEU on the WMT 2014 English-to-German
    translation task, improving over the existing best results, including
    ensembles, by over 2 BLEU. On the WMT 2014 English-to-French translation
    task, our model establishes a new single-model state-of-the-art BLEU
    score of 41.8 after training for 3.5 days on eight GPUs, a small
    fraction of the training costs of the best models from the literature.
    """,
    "url": "https://arxiv.org/abs/1706.03762",
    "source_name": "arXiv",
    "source_type": "paper"
}

sample_news = {
    "title": "OpenAI Announces GPT-4",
    "content": """
    OpenAI today announced GPT-4, the latest milestone in its effort
    to scale up deep learning. GPT-4 is a large multimodal model
    that can accept image and text inputs and produce text outputs.
    While less capable than humans in many real-world scenarios,
    GPT-4 exhibits human-level performance on various professional
    and academic benchmarks.
    """,
    "url": "https://openai.com/research/gpt-4",
    "source_name": "OpenAI",
    "source_type": "news"
}

sample_blog = {
    "title": "BERT: Pre-training of Deep Bidirectional Transformers",
    "content": """
    We introduce a new language representation model called BERT,
    which stands for Bidirectional Encoder Representations from Transformers.
    Unlike recent language representation models, BERT is designed to
    pre-train deep bidirectional representations from unlabeled text
    by jointly conditioning on both left and right context in all layers.
    """,
    "url": "https://arxiv.org/abs/1810.04805",
    "source_name": "arXiv",
    "source_type": "paper"
}

## 0. 서버 상태 확인

In [ ]:
# Health check
try:
    response = requests.get(f"{BASE_URL}/health")
    print(f"✅ Server Status: {response.status_code}")
    if response.status_code == 200:
        pprint(response.json())
except requests.exceptions.ConnectionError:
    print("❌ Server is not running!")
    print("Please start the server: uvicorn src.app.api.main:app --reload")

## 1. POST /processors/summarize - 아티클 요약

### 1.1 한국어 요약 (medium)

In [ ]:
payload = {
    "title": sample_paper["title"],
    "content": sample_paper["content"],
    "language": "ko",
    "length": "medium"
}

response = requests.post(f"{PROCESSORS_URL}/summarize", json=payload)
print(f"Status Code: {response.status_code}\n")

if response.status_code == 200:
    result = response.json()
    print(f"✅ 한국어 요약 성공")
    print(f"\nTitle: {result['title']}")
    print(f"Summary: {result['summary']}")
    print(f"Language: {result['language']}")
    print(f"Length: {result['length']}")
else:
    print(f"❌ Error: {response.text}")

### 1.2 영어 요약 (short)

In [ ]:
payload = {
    "title": sample_paper["title"],
    "content": sample_paper["content"],
    "language": "en",
    "length": "short"
}

response = requests.post(f"{PROCESSORS_URL}/summarize", json=payload)
print(f"Status Code: {response.status_code}\n")

if response.status_code == 200:
    result = response.json()
    print(f"✅ 영어 요약 성공")
    print(f"\nSummary: {result['summary']}")
else:
    print(f"❌ Error: {response.text}")

### 1.3 한국어 요약 (long)

In [ ]:
payload = {
    "title": sample_paper["title"],
    "content": sample_paper["content"],
    "language": "ko",
    "length": "long"
}

response = requests.post(f"{PROCESSORS_URL}/summarize", json=payload)
print(f"Status Code: {response.status_code}\n")

if response.status_code == 200:
    result = response.json()
    print(f"✅ 긴 요약 성공")
    print(f"\nSummary length: {len(result['summary'])} chars")
    print(f"Summary: {result['summary'][:200]}...")
else:
    print(f"❌ Error: {response.text}")

## 2. POST /processors/evaluate - 중요도 평가

### 2.1 기본 평가 (메타데이터 없음)

In [ ]:
payload = {
    "title": sample_paper["title"],
    "content": sample_paper["content"]
}

response = requests.post(f"{PROCESSORS_URL}/evaluate", json=payload)
print(f"Status Code: {response.status_code}\n")

if response.status_code == 200:
    result = response.json()
    print(f"✅ 중요도 평가 성공")
    print(f"\nTitle: {result['title']}")
    print(f"Final Score: {result['final_score']:.3f}")
    print(f"Innovation: {result['innovation']:.3f}")
    print(f"Relevance: {result['relevance']:.3f}")
    print(f"Impact: {result['impact']:.3f}")
    print(f"Timeliness: {result['timeliness']:.3f}")
    print(f"\nReasoning: {result['reasoning'][:200]}...")
else:
    print(f"❌ Error: {response.text}")

### 2.2 메타데이터 포함 평가

In [ ]:
payload = {
    "title": sample_paper["title"],
    "content": sample_paper["content"],
    "metadata": {
        "year": 2017,
        "citations": 50000,
        "venue": "NeurIPS"
    }
}

response = requests.post(f"{PROCESSORS_URL}/evaluate", json=payload)
print(f"Status Code: {response.status_code}\n")

if response.status_code == 200:
    result = response.json()
    print(f"✅ 메타데이터 포함 평가 성공")
    print(f"\nFinal Score: {result['final_score']:.3f}")
    print(f"Impact: {result['impact']:.3f} (높은 인용 수 반영)")
else:
    print(f"❌ Error: {response.text}")

### 2.3 뉴스 아티클 평가

In [ ]:
payload = {
    "title": sample_news["title"],
    "content": sample_news["content"],
    "metadata": {
        "published_date": "2023-03-14",
        "source": "OpenAI"
    }
}

response = requests.post(f"{PROCESSORS_URL}/evaluate", json=payload)
print(f"Status Code: {response.status_code}\n")

if response.status_code == 200:
    result = response.json()
    print(f"✅ 뉴스 평가 성공")
    print(f"\nFinal Score: {result['final_score']:.3f}")
    print(f"Timeliness: {result['timeliness']:.3f} (최신성 반영)")
else:
    print(f"❌ Error: {response.text}")

## 3. POST /processors/classify - 카테고리 분류

### 3.1 논문 분류

In [ ]:
payload = {
    "title": sample_paper["title"],
    "content": sample_paper["content"],
    "source_name": sample_paper["source_name"],
    "url": sample_paper["url"]
}

response = requests.post(f"{PROCESSORS_URL}/classify", json=payload)
print(f"Status Code: {response.status_code}\n")

if response.status_code == 200:
    result = response.json()
    print(f"✅ 논문 분류 성공")
    print(f"\nTitle: {result['title']}")
    print(f"Category: {result['category']}")
    print(f"Confidence: {result['confidence']:.3f}")
    print(f"Research Field: {result['research_field']}")
    print(f"Sub-fields: {', '.join(result['sub_fields'])}")
    print(f"Keywords: {', '.join(result['keywords'][:10])}")
    print(f"\nReasoning: {result['reasoning'][:200]}...")
else:
    print(f"❌ Error: {response.text}")

### 3.2 뉴스 분류

In [ ]:
payload = {
    "title": sample_news["title"],
    "content": sample_news["content"],
    "source_name": sample_news["source_name"],
    "url": sample_news["url"]
}

response = requests.post(f"{PROCESSORS_URL}/classify", json=payload)
print(f"Status Code: {response.status_code}\n")

if response.status_code == 200:
    result = response.json()
    print(f"✅ 뉴스 분류 성공")
    print(f"\nCategory: {result['category']}")
    print(f"Confidence: {result['confidence']:.3f}")
    print(f"Keywords: {', '.join(result['keywords'][:10])}")
else:
    print(f"❌ Error: {response.text}")

### 3.3 최소 정보로 분류 (Fallback 테스트)

In [ ]:
payload = {
    "title": "Some Random Article",
    "content": "This is a very short article with minimal information.",
    "source_name": "",
    "url": ""
}

response = requests.post(f"{PROCESSORS_URL}/classify", json=payload)
print(f"Status Code: {response.status_code}\n")

if response.status_code == 200:
    result = response.json()
    print(f"✅ Fallback 분류 성공")
    print(f"\nCategory: {result['category']} (likely 'other')")
    print(f"Confidence: {result['confidence']:.3f}")
else:
    print(f"❌ Error: {response.text}")

## 4. POST /processors/process - 전체 파이프라인 처리

### 4.1 논문 전체 처리

In [ ]:
payload = {
    "title": sample_paper["title"],
    "content": sample_paper["content"],
    "url": sample_paper["url"],
    "source_name": sample_paper["source_name"],
    "source_type": sample_paper["source_type"],
    "metadata": {
        "year": 2017,
        "citations": 50000
    },
    "summary_length": "medium",
    "summary_language": "ko"
}

start_time = time.time()
response = requests.post(f"{PROCESSORS_URL}/process", json=payload)
elapsed = time.time() - start_time

print(f"Status Code: {response.status_code}")
print(f"Processing Time: {elapsed:.2f}s\n")

if response.status_code == 200:
    result = response.json()
    print(f"✅ 전체 파이프라인 처리 성공")
    print(f"\nTitle: {result['title']}")
    print(f"Summary: {result['summary'][:150]}...")
    print(f"\nImportance Score: {result['importance_score']:.3f}")
    print(f"  - Innovation: {result['innovation_score']:.3f}")
    print(f"  - Relevance: {result['relevance_score']:.3f}")
    print(f"  - Impact: {result['impact_score']:.3f}")
    print(f"  - Timeliness: {result['timeliness_score']:.3f}")
    print(f"\nCategory: {result['category']}")
    print(f"Research Field: {result['research_field']}")
    print(f"Keywords: {', '.join(result['keywords'][:10])}")
    print(f"\nEmbedding Dimensions: {len(result['embedding'])}")
    print(f"Embedding Sample: {result['embedding'][:5]}")
    print(f"\nProcessed At: {result['processed_at']}")
else:
    print(f"❌ Error: {response.text}")

### 4.2 뉴스 전체 처리 (영어 요약)

In [ ]:
payload = {
    "title": sample_news["title"],
    "content": sample_news["content"],
    "url": sample_news["url"],
    "source_name": sample_news["source_name"],
    "source_type": sample_news["source_type"],
    "summary_length": "short",
    "summary_language": "en"
}

start_time = time.time()
response = requests.post(f"{PROCESSORS_URL}/process", json=payload)
elapsed = time.time() - start_time

print(f"Status Code: {response.status_code}")
print(f"Processing Time: {elapsed:.2f}s\n")

if response.status_code == 200:
    result = response.json()
    print(f"✅ 뉴스 전체 처리 성공")
    print(f"\nSummary (EN): {result['summary']}")
    print(f"Category: {result['category']}")
    print(f"Importance Score: {result['importance_score']:.3f}")
else:
    print(f"❌ Error: {response.text}")

## 5. POST /processors/batch-process - 배치 처리

### 5.1 3개 아티클 배치 처리

In [ ]:
payload = {
    "articles": [
        {
            "title": sample_paper["title"],
            "content": sample_paper["content"],
            "url": sample_paper["url"],
            "source_name": sample_paper["source_name"],
            "metadata": {"year": 2017, "citations": 50000}
        },
        {
            "title": sample_news["title"],
            "content": sample_news["content"],
            "url": sample_news["url"],
            "source_name": sample_news["source_name"],
            "metadata": {"published_date": "2023-03-14"}
        },
        {
            "title": sample_blog["title"],
            "content": sample_blog["content"],
            "url": sample_blog["url"],
            "source_name": sample_blog["source_name"],
            "metadata": {"year": 2018, "citations": 30000}
        }
    ],
    "max_concurrent": 3,
    "summary_length": "medium",
    "summary_language": "ko"
}

start_time = time.time()
response = requests.post(f"{PROCESSORS_URL}/batch-process", json=payload)
elapsed = time.time() - start_time

print(f"Status Code: {response.status_code}")
print(f"Total Processing Time: {elapsed:.2f}s\n")

if response.status_code == 200:
    result = response.json()
    print(f"✅ 배치 처리 성공")
    print(f"\nTotal Processed: {result['total']}")
    print(f"Succeeded: {result['succeeded']}")
    print(f"Failed: {result['failed']}")
    print(f"Success Rate: {result['success_rate']:.1f}%")
    print(f"Processing Time: {result['processing_time']:.2f}s")
    
    print(f"\n{'='*80}")
    print("Processed Articles:")
    print(f"{'='*80}")
    
    for i, article in enumerate(result['articles'], 1):
        print(f"\n[{i}] {article['title'][:50]}...")
        print(f"    Category: {article['category']}")
        print(f"    Importance: {article['importance_score']:.3f}")
        print(f"    Research Field: {article['research_field']}")
        print(f"    Keywords: {', '.join(article['keywords'][:5])}")
        print(f"    Summary: {article['summary'][:100]}...")
else:
    print(f"❌ Error: {response.text}")

### 5.2 배치 처리 (max_concurrent=2)

In [ ]:
payload = {
    "articles": [
        {
            "title": sample_paper["title"],
            "content": sample_paper["content"],
            "url": sample_paper["url"],
            "source_name": sample_paper["source_name"]
        },
        {
            "title": sample_news["title"],
            "content": sample_news["content"],
            "url": sample_news["url"],
            "source_name": sample_news["source_name"]
        },
        {
            "title": sample_blog["title"],
            "content": sample_blog["content"],
            "url": sample_blog["url"],
            "source_name": sample_blog["source_name"]
        }
    ],
    "max_concurrent": 2,
    "summary_length": "short",
    "summary_language": "en"
}

start_time = time.time()
response = requests.post(f"{PROCESSORS_URL}/batch-process", json=payload)
elapsed = time.time() - start_time

print(f"Status Code: {response.status_code}")
print(f"Total Processing Time: {elapsed:.2f}s (limited to 2 concurrent)\n")

if response.status_code == 200:
    result = response.json()
    print(f"✅ 제한된 동시성 배치 처리 성공")
    print(f"\nSucceeded: {result['succeeded']}/{result['total']}")
    print(f"Processing Time: {result['processing_time']:.2f}s")
else:
    print(f"❌ Error: {response.text}")

## 6. POST /processors/statistics - 통계 계산

### 6.1 배치 처리 결과로 통계 계산

In [ ]:
# 먼저 배치 처리로 데이터 생성
batch_payload = {
    "articles": [
        {"title": sample_paper["title"], "content": sample_paper["content"], "url": sample_paper["url"], "source_name": sample_paper["source_name"]},
        {"title": sample_news["title"], "content": sample_news["content"], "url": sample_news["url"], "source_name": sample_news["source_name"]},
        {"title": sample_blog["title"], "content": sample_blog["content"], "url": sample_blog["url"], "source_name": sample_blog["source_name"]}
    ],
    "max_concurrent": 3,
    "summary_length": "medium",
    "summary_language": "ko"
}

batch_response = requests.post(f"{PROCESSORS_URL}/batch-process", json=batch_payload)

if batch_response.status_code == 200:
    batch_result = batch_response.json()
    
    # 통계 계산 요청
    stats_payload = {
        "processed_articles": batch_result["articles"]
    }
    
    response = requests.post(f"{PROCESSORS_URL}/statistics", json=stats_payload)
    print(f"Status Code: {response.status_code}\n")
    
    if response.status_code == 200:
        result = response.json()
        print(f"✅ 통계 계산 성공")
        print(f"\nTotal Articles: {result['total']}")
        print(f"\nScore Statistics:")
        print(f"  - Average Score: {result['average_score']:.3f}")
        print(f"  - Max Score: {result['max_score']:.3f}")
        print(f"  - Min Score: {result['min_score']:.3f}")
        print(f"  - High Quality Count (≥0.7): {result['high_quality_count']}")
        print(f"\nCategory Distribution:")
        for category, count in result['category_distribution'].items():
            print(f"  - {category}: {count}")
    else:
        print(f"❌ Error: {response.text}")
else:
    print(f"❌ Batch processing failed: {batch_response.text}")

### 6.2 빈 리스트 통계 테스트

In [ ]:
payload = {
    "processed_articles": []
}

response = requests.post(f"{PROCESSORS_URL}/statistics", json=payload)
print(f"Status Code: {response.status_code}\n")

if response.status_code == 200:
    result = response.json()
    print(f"✅ 빈 리스트 통계 처리 성공")
    print(f"\nResult: {result}")
    print(f"Expected: Empty dictionary or zero values")
else:
    print(f"❌ Error: {response.text}")

## 7. 에러 처리 테스트

### 7.1 필수 필드 누락 (title 없음)

In [ ]:
payload = {
    "content": sample_paper["content"],
    "language": "ko",
    "length": "medium"
}

response = requests.post(f"{PROCESSORS_URL}/summarize", json=payload)
print(f"Status Code: {response.status_code}\n")

if response.status_code == 422:
    print(f"✅ 예상된 유효성 검사 오류 발생")
    print(f"Error Detail: {response.json()}")
else:
    print(f"❌ Unexpected response: {response.text}")

### 7.2 잘못된 language 값

In [ ]:
payload = {
    "title": sample_paper["title"],
    "content": sample_paper["content"],
    "language": "invalid_language",
    "length": "medium"
}

response = requests.post(f"{PROCESSORS_URL}/summarize", json=payload)
print(f"Status Code: {response.status_code}\n")

if response.status_code == 422:
    print(f"✅ 예상된 유효성 검사 오류 발생 (잘못된 language)")
    error_detail = response.json()
    print(f"Error Detail: {error_detail}")
else:
    print(f"❌ Unexpected response: {response.text}")

### 7.3 빈 content 처리

In [ ]:
payload = {
    "title": "Empty Article",
    "content": "",
    "language": "ko",
    "length": "medium"
}

response = requests.post(f"{PROCESSORS_URL}/summarize", json=payload)
print(f"Status Code: {response.status_code}\n")

if response.status_code in [400, 422, 500]:
    print(f"✅ 빈 content에 대한 오류 처리 확인")
    print(f"Error: {response.json()}")
elif response.status_code == 200:
    print(f"⚠️ 빈 content가 처리됨 (의도된 동작인지 확인 필요)")
    print(f"Result: {response.json()}")
else:
    print(f"❌ Unexpected response: {response.text}")

## 8. 성능 벤치마크

### 8.1 요약 성능 비교 (short vs medium vs long)

In [ ]:
print("요약 길이별 성능 벤치마크\n" + "="*80)

lengths = ["short", "medium", "long"]
results = {}

for length in lengths:
    payload = {
        "title": sample_paper["title"],
        "content": sample_paper["content"],
        "language": "ko",
        "length": length
    }
    
    start_time = time.time()
    response = requests.post(f"{PROCESSORS_URL}/summarize", json=payload)
    elapsed = time.time() - start_time
    
    if response.status_code == 200:
        result = response.json()
        summary_length = len(result['summary'])
        results[length] = {"time": elapsed, "chars": summary_length}
        print(f"\n{length.upper():8s}: {elapsed:.2f}s | {summary_length:4d} chars")
    else:
        print(f"\n{length.upper():8s}: Failed - {response.text}")

print("\n" + "="*80)
print("✅ 요약 성능 벤치마크 완료")

### 8.2 배치 처리 성능 (동시성 비교)

In [ ]:
print("배치 처리 동시성 성능 벤치마크\n" + "="*80)

articles = [
    {"title": sample_paper["title"], "content": sample_paper["content"], "url": sample_paper["url"], "source_name": sample_paper["source_name"]},
    {"title": sample_news["title"], "content": sample_news["content"], "url": sample_news["url"], "source_name": sample_news["source_name"]},
    {"title": sample_blog["title"], "content": sample_blog["content"], "url": sample_blog["url"], "source_name": sample_blog["source_name"]}
]

concurrency_levels = [1, 2, 3]
benchmark_results = {}

for max_concurrent in concurrency_levels:
    payload = {
        "articles": articles,
        "max_concurrent": max_concurrent,
        "summary_length": "short",
        "summary_language": "ko"
    }
    
    start_time = time.time()
    response = requests.post(f"{PROCESSORS_URL}/batch-process", json=payload)
    elapsed = time.time() - start_time
    
    if response.status_code == 200:
        result = response.json()
        benchmark_results[max_concurrent] = elapsed
        print(f"\nConcurrency {max_concurrent}: {elapsed:.2f}s | Success: {result['succeeded']}/{result['total']}")
    else:
        print(f"\nConcurrency {max_concurrent}: Failed - {response.text}")

print("\n" + "="*80)
if len(benchmark_results) > 1:
    speedup_2 = benchmark_results[1] / benchmark_results[2] if 2 in benchmark_results else 0
    speedup_3 = benchmark_results[1] / benchmark_results[3] if 3 in benchmark_results else 0
    print(f"Speedup (2 vs 1): {speedup_2:.2f}x")
    print(f"Speedup (3 vs 1): {speedup_3:.2f}x")
print("✅ 배치 처리 성능 벤치마크 완료")

## 9. End-to-End 워크플로우 테스트

### 9.1 전체 워크플로우: 요약 → 평가 → 분류 → 통합 처리

In [ ]:
print("End-to-End 워크플로우 테스트\n" + "="*80)

test_article = sample_paper.copy()

# Step 1: 요약
print("\n[Step 1] 요약 생성...")
summary_payload = {
    "title": test_article["title"],
    "content": test_article["content"],
    "language": "ko",
    "length": "medium"
}
summary_response = requests.post(f"{PROCESSORS_URL}/summarize", json=summary_payload)
if summary_response.status_code == 200:
    summary = summary_response.json()["summary"]
    print(f"✅ 요약: {summary[:100]}...")
else:
    print(f"❌ 요약 실패")
    summary = None

# Step 2: 평가
print("\n[Step 2] 중요도 평가...")
eval_payload = {
    "title": test_article["title"],
    "content": test_article["content"],
    "metadata": {"year": 2017, "citations": 50000}
}
eval_response = requests.post(f"{PROCESSORS_URL}/evaluate", json=eval_payload)
if eval_response.status_code == 200:
    eval_result = eval_response.json()
    print(f"✅ 중요도 점수: {eval_result['final_score']:.3f}")
else:
    print(f"❌ 평가 실패")
    eval_result = None

# Step 3: 분류
print("\n[Step 3] 카테고리 분류...")
classify_payload = {
    "title": test_article["title"],
    "content": test_article["content"],
    "source_name": test_article["source_name"],
    "url": test_article["url"]
}
classify_response = requests.post(f"{PROCESSORS_URL}/classify", json=classify_payload)
if classify_response.status_code == 200:
    classify_result = classify_response.json()
    print(f"✅ 카테고리: {classify_result['category']} ({classify_result['research_field']})")
else:
    print(f"❌ 분류 실패")
    classify_result = None

# Step 4: 통합 처리 (모든 단계 한번에)
print("\n[Step 4] 통합 파이프라인 처리...")
process_payload = {
    "title": test_article["title"],
    "content": test_article["content"],
    "url": test_article["url"],
    "source_name": test_article["source_name"],
    "source_type": test_article["source_type"],
    "metadata": {"year": 2017, "citations": 50000},
    "summary_length": "medium",
    "summary_language": "ko"
}
process_response = requests.post(f"{PROCESSORS_URL}/process", json=process_payload)
if process_response.status_code == 200:
    process_result = process_response.json()
    print(f"✅ 통합 처리 완료")
    print(f"   - Summary: {process_result['summary'][:100]}...")
    print(f"   - Score: {process_result['importance_score']:.3f}")
    print(f"   - Category: {process_result['category']}")
    print(f"   - Embedding: {len(process_result['embedding'])} dims")
else:
    print(f"❌ 통합 처리 실패")

print("\n" + "="*80)
print("✅ End-to-End 워크플로우 테스트 완료")

## 10. 테스트 요약

### 전체 테스트 결과

In [ ]:
print("\n" + "="*80)
print("✅ Processor API 테스트 완료")
print("="*80)
print("""
테스트 완료된 엔드포인트:
  1. POST /processors/summarize (한국어/영어, short/medium/long)
  2. POST /processors/evaluate (메타데이터 포함/미포함)
  3. POST /processors/classify (논문/뉴스/블로그)
  4. POST /processors/process (전체 파이프라인)
  5. POST /processors/batch-process (배치 처리)
  6. POST /processors/statistics (통계 계산)
  7. 에러 처리 (필수 필드 누락, 잘못된 값)
  8. 성능 벤치마크 (요약 길이별, 동시성별)
  9. End-to-End 워크플로우

모든 테스트가 정상적으로 완료되었습니다! 🎉
""")
print("="*80)